# Customer Segmentation Using Data Analysis & Clustering

## QSkill Data Science Internship – Task 3

### Objective
The aim of this project is to analyze customer purchasing behavior, calculate RFM (Recency, Frequency and Monetary) values, and group customers into different segments using K-Means clustering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

url = "https://raw.githubusercontent.com/sheezanazeer98/Online-Retail-SQL/main/Online%20Retail.csv"
df = pd.read_csv(url, encoding="ISO-8859-1")

df.head()

## 1. Understanding the Dataset

In [ ]:
print("Shape of dataset:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
df.info()

## 2. Checking Missing Values and Duplicates

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

## 3. Data Cleaning

In [ ]:
df = df.dropna(subset=["CustomerID"])
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df = df.drop_duplicates()
df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]

print("Shape after cleaning:", df.shape)

## 4. Basic Sales Analysis

In [ ]:
print("Total customers:", df["CustomerID"].nunique())
print("Total invoices:", df["InvoiceNo"].nunique())
print("Total sales:", round(df["TotalAmount"].sum(), 2))
df[["Quantity", "UnitPrice", "TotalAmount"]].describe()

## 5. Monthly Sales Trend

In [ ]:
monthly_sales = df.groupby(df["InvoiceDate"].dt.to_period("M"))["TotalAmount"].sum()

plt.figure(figsize=(10,5))
monthly_sales.plot()
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.show()

## 6. Creating RFM Analysis

In [ ]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg({
    "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
    "InvoiceNo": "nunique",
    "TotalAmount": "sum"
})

rfm.rename(columns={
    "InvoiceDate": "Recency",
    "InvoiceNo": "Frequency",
    "TotalAmount": "Monetary"
}, inplace=True)

rfm.head()

## 7. RFM Summary

In [ ]:
print(rfm.describe())
print("\nNumber of customers:", len(rfm))

## 8. RFM Visualizations

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(rfm["Recency"], bins=30)
plt.title("Customer Recency")
plt.xlabel("Days Since Last Purchase")
plt.show()

plt.figure(figsize=(8,5))
sns.histplot(rfm["Frequency"], bins=30)
plt.title("Customer Frequency")
plt.xlabel("Number of Purchases")
plt.show()

plt.figure(figsize=(8,5))
sns.histplot(rfm["Monetary"], bins=30)
plt.title("Customer Monetary Value")
plt.xlabel("Total Spending")
plt.show()

## 9. Preparing Data for K-Means

In [ ]:
rfm_log = np.log1p(rfm)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

rfm_scaled = pd.DataFrame(rfm_scaled, index=rfm.index, columns=rfm.columns)
rfm_scaled.head()

## 10. Finding the Number of Clusters

In [ ]:
from sklearn.cluster import KMeans

inertia = []

for k in range(2, 7):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(rfm_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8,5))
plt.plot(range(2, 7), inertia, marker="o")
plt.title("Elbow Method")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.show()

## 11. Applying K-Means Clustering

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)
rfm.head()

## 12. Checking Cluster Quality

In [ ]:
from sklearn.metrics import silhouette_score

score = silhouette_score(rfm_scaled, rfm["Cluster"])
print("Silhouette Score:", round(score, 3))

## 13. Customer Segment Analysis

In [ ]:
cluster_summary = rfm.groupby("Cluster")[["Recency", "Frequency", "Monetary"]].mean().round(2)
print(cluster_summary)

## 14. Number of Customers in Each Segment

In [ ]:
cluster_counts = rfm["Cluster"].value_counts().sort_index()

plt.figure(figsize=(8,5))
cluster_counts.plot(kind="bar")
plt.title("Number of Customers in Each Segment")
plt.xlabel("Cluster")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.show()

## 15. Customer Segments

In [ ]:
cluster_means = rfm.groupby("Cluster")[["Recency", "Frequency", "Monetary"]].mean()

best_cluster = cluster_means["Monetary"].idxmax()
recent_cluster = cluster_means["Recency"].idxmin()
low_cluster = cluster_means["Monetary"].idxmin()

segment_names = {}

for cluster in cluster_means.index:
    if cluster == best_cluster:
        segment_names[cluster] = "High Value Customers"
    elif cluster == recent_cluster:
        segment_names[cluster] = "Recent Customers"
    elif cluster == low_cluster:
        segment_names[cluster] = "Low Value Customers"
    else:
        segment_names[cluster] = "Regular Customers"

rfm["Segment"] = rfm["Cluster"].map(segment_names)
print(rfm["Segment"].value_counts())

## 16. Business Recommendations

In [ ]:
print("High Value Customers: Focus on loyalty rewards and personalized offers.")
print("Recent Customers: Encourage repeat purchases with follow-up offers.")
print("Regular Customers: Use promotions to increase purchase frequency.")
print("Low Value Customers: Use targeted discounts and re-engagement campaigns.")

## Conclusion

RFM analysis was used to understand customer purchasing behavior based on Recency, Frequency and Monetary value. K-Means clustering was then applied to group customers with similar purchasing patterns. The resulting customer segments can help a business plan targeted marketing, loyalty and customer retention strategies.